# Imports, Settings, & Functions

In [14]:
import os
import gc
from glob import glob
import joblib
import numpy as np
import pandas as pd

# Scipy
from scipy.signal import butter, filtfilt, iirnotch, hilbert
from scipy.stats import kurtosis
from scipy.io import savemat 

# Scikit-Learn
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

# Pytorch
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
from torch.utils.data import random_split
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau


In [15]:
# Noise Filters
def butter_bandpass(lowcut, highcut, fs, order=5):
    nyq = fs / 2.0
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    return b, a

def bandpass_filter(data, lowcut=1.0, highcut=200.0, fs=1000.0, order=4):
    b, a = butter_bandpass(lowcut, highcut, fs, order=order)
    return filtfilt(b, a, data, axis=0)

# Apply after bandpass
def notch_filter(data, freq=60.0, fs=1000.0, quality=30.0):
    b, a = iirnotch(freq, quality, fs)
    return filtfilt(b, a, data, axis=0)

# Noise Metrics for evaluation
def compute_rmse(true, estimate):
    return np.sqrt(np.mean((true - estimate) ** 2))

# Kurtosis signal reduction > 0 shows a denoised signal
def proportion_of_positive_kurtosis_signals(kurtosis_raw, kurtosis_denoised):
    return (np.array([(kurtosis_raw - kurtosis_denoised) > 0]).sum() / len(kurtosis_raw)) * 100

# Use a Standard scaler to reduce the mean to 0 and std to 1

In [ ]:
# Computing the power envelope of each channel

def band_power_envelope(ecog_signal: np.ndarray, lowcut: float, highcut: float, fs: float = 1000.0, order: int = 4) -> np.ndarray:
    """Computes band-limited envelope via Hilbert transform.
    Parameters
    ----------
    self.ecog_signal : np.ndarray (T, channels)
        This is the ecog signal that has been filtered.
    lowcut : float
        This is the lower band limit in Hz.
    highcut : float
        This is the upper band limit in Hz.
    fs : float, optional
        This is the frequency of the sample., by default 1000.0
    order : int, optional
        This is the Butterworth order, by default 4
    Returns
    -------
    np.ndarray
        envelope
    """
    # 1. Narrowband bandpass
    b, a = butter_bandpass(lowcut, highcut, fs, order=order)
    narrow = filtfilt(b, a, ecog_signal, axis=0)
    # 2. Hilbert transform to get analytic signal
    analytic = hilbert(narrow, axis=0)
    # 3. Envelope = absolute value
    envelope = np.abs(analytic)
    return envelope

def multiband_features(ecog_raw: np.ndarray, fs: float = 1000.0) -> np.ndarray:
    """Builds concatenated band-power features for μ, β, and high-gamma using a Hilbert transform.
    Parameters
    ----------
    ecog_raw : np.ndarray
        (T, 64)
    fs : float, optional
        Frequency of the sample, by default 1000.0
    Returns
    -------
    np.ndarray
        features: (T, 64, 3) (μ, β, high-gamma per electrode)
    """
    mu_env = band_power_envelope(ecog_raw, lowcut=8.0, highcut=13.0, fs=fs)
    beta_env = band_power_envelope(ecog_raw, lowcut=13.0, highcut=30.0, fs=fs)
    hg_env = band_power_envelope(ecog_raw, lowcut=70.0, highcut=200.0, fs=fs)
    # Concatenate along channel dimension
    return np.concatenate([mu_env, beta_env, hg_env], axis=1)


In [ ]:
def create_overlapping_windows(ecog_values: np.ndarray, motion_values: np.ndarray, window_size: int = 20, hop_size: int = 10):
    """Builds overlapping windows to increase sample count and capture smoother transitions.

    Parameters
    ----------
    ecog_values : np.ndarray
        (T, features)
    motion_values : np.ndarray
        (T_motion, 3)_
    window_size : int, optional
        number of timepoints per window, by default 20
    hop_size : int, optional
        step bewteen windows, by default 10
    """
    num_samples, num_features = ecog_values.shape
    print(f"number of Samples")
    max_windows = (num_samples - window_size) // hop_size + 1
    X_list = []
    y_list = []
    for w in range(max_windows):
        start = w * hop_size
        end = start + window_size
        if end > num_samples:
            break
        # Assign label as motion at center of window (or last timepoint)
        X_list.append(ecog_values[start:end, :])
        y_list.append(motion_values[min(end -1, motion_values.shape[0] -1), :])
    X = np.stack(X_list, axis=0)
    y = np.stack(y_list, axis=0)
    return X, y        


In [18]:
def predict_and_export(model, data_loader, device, output_file_path):
    model.eval()
    all_preds, all_targets = [], []

    with torch.no_grad():
        for inputs, targets in data_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            all_preds.append(outputs.cpu().numpy())
            all_targets.append(targets.cpu().numpy())
    predictions = np.concatenate(all_preds, axis=0)
    targets = np.concatenate(all_targets, axis=0)
    
    # Save as .mat file for visualization
    savemat(output_file_path, {
        "predictions":predictions,
        "targets": targets
    })
    print("Saved predictions to ecog_predictions.mat")

    return predictions, targets

In [ ]:
# Defining Preprocessing for the raw data
class PreprocessData:
    def __init__(self, ecog_file_path, motion_file_path):
        self.ecog_file_path = ecog_file_path
        self.motion_file_path = motion_file_path
        self.ecog_data = None
        self.motion_data = None
        self.filtered_ecog = None
        self.scaled_ecog = None
        self.X = None
        self.y = None
        self.scaler = None

    def process(self, eval=False, window_size=20, duration_limit=900):
        self.read_data()
        self.common_average_reference()
        self.filter_signal(eval=eval)
        self.format_data(window_size=window_size, duration_limit=duration_limit)
        return self.X, self.y
    
    def read_data(self):
        print("Reading data")
        self.ecog_data = pd.read_csv(self.ecog_file_path)
        self.motion_data = pd.read_csv(self.motion_file_path)
        print(f"self.ecog_data.shape:{self.ecog_data.shape}")
        print(f"self.motion_data.shape:{self.motion_data.shape}")
        return self

    def common_average_reference(self):
        # Subtract the common mean from the signals 
        print("Subtracting common mean from the signals to create common average reference")
        common_average_reference = np.mean(self.ecog_data.drop(["Time", "Fs"], axis=1).values, axis=1, keepdims=1)
        ecog_data_values = self.ecog_data[self.ecog_data.columns[1:-1]].values
        ecog_data_common_mean_subtracted = ecog_data_values - common_average_reference
        self.ecog_data[self.ecog_data.columns[1:-1]] = ecog_data_common_mean_subtracted
        del ecog_data_values, ecog_data_common_mean_subtracted, common_average_reference
        gc.collect()
        return self

    def filter_signal(self, eval=False):
        ecog_raw = self.ecog_data[self.ecog_data.columns[1:-1]].values
        print(f"Raw Data Shape: {ecog_raw.shape}")

        # Apply filters
        print(f"Applying a bandpass filter from 1 KHz to 200 KHz")
        filtered = bandpass_filter(ecog_raw, lowcut=1.0, highcut=200.0, fs=1000.0, order=4)
        print(f"Removing 60 Hz Electrical Noise with a Notch Filter")
        denoised = notch_filter(filtered, freq=60, fs=1000.0)
        print(f"Denoised Shape: {denoised.shape}")
        # Evaluate filters
        if eval:
            kurt_raw = kurtosis(ecog_raw, axis=0, fisher=True)
            kurt_denoised = kurtosis(denoised, axis=0, fisher=True)
            proportion_of_positive_kurtosis_signals(kurt_raw, kurt_denoised)
            compute_rmse(ecog_raw, denoised)

        # Compute Power Envelopes
        print("Computing Power Envelopes: Builds concatenated band-power features for μ, β, and high-gamma using a Hilbert transform")
        features = multiband_features(denoised, fs=1000.0) # shape (T, 192)
        print(f"Features Shape of the Multiband Features: {features.shape}")

        # Identify the principal components of the network
        print(f"Identifying 64 Principal components of the network")
        pca = PCA(n_components = 64, random_state=42)
        reduced = pca.fit_transform(features)
        print(f"Reduced Shape from PCA: {reduced.shape}")
        # Scale
        print(f"Scaling the data to have a mean of 0 and standard deviation of 1")
        self.scaler = StandardScaler()
        self.scaled_ecog = self.scaler.fit_transform(reduced)

        # Replace in DataFrame
        self.ecog_data = self.ecog_data.copy()
        self.ecog_data[self.ecog_data.columns[1:-1]] = self.scaled_ecog

        # Clean memory
        del ecog_raw, filtered, denoised
        gc.collect()
        return self

    def format_data(self, window_size=20, duration_limit=900):
        print(f"This data has been preprocessed.")
        print(f"Truncating the data to have the same 15 minute limit.")
        ecog_df = self.ecog_data[self.ecog_data["Time"] <= duration_limit]
        motion_df = self.motion_data[self.motion_data["Motion_time"] <= duration_limit]

        ecog_values = ecog_df.drop(columns=["Fs", "Time"]).values
        motion_values = motion_df.drop(columns=["Fsm", "Motion_time"]).values

        print(f"motion_values.shape: {motion_values.shape}")
        print(f"ecog_values.shape: {ecog_values.shape}")

        # Smooth the signal
        print(f"Creating Overlapping Windows of data to Smooth the Signal")
        X, y = create_overlapping_windows(ecog_values, motion_values, window_size=20, hop_size=10)
        print(f"y.shape: {y.shape}")
        self.X, self.y = X, y
        
        print(self.X.shape)
        print(self.y.shape)
        
        # Clean up
        del ecog_values, motion_values
        gc.collect()

    def save(self):
        output_file_path_base = self.ecog_file_path.strip("ecog_data.csv")
        joblib.dump(self.scaler, output_file_path_base + "scaler_ecog.pkl")
        np.save(output_file_path_base + "X.npy", self.X)
        np.save(output_file_path_base + "y.npy", self.y)
        


In [20]:
# Model definitions
class EcogMotionDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]
    
# CNN/LSTM hybrid
class EcogToMotionNet(nn.Module):
    def __init__(self):
        super().__init__()

        # CNN component: outputs 256 channels
        self.convolv = nn.Sequential(
            nn.Conv1d(in_channels=64, out_channels=128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Conv1d(in_channels=128, out_channels=128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Conv1d(in_channels=128, out_channels=256, kernel_size=3, padding=1),  # Fixed to 256 channels
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.3)
        )

        # Bi-LSTM component (2 Layers)
        self.lstm = nn.LSTM(input_size=256, hidden_size=128, num_layers=2, batch_first=True, bidirectional=True)

        self.attn_weight = nn.Linear(2 * 128, 1, bias=False)

        # Fully connected layer
        self.fc = nn.Sequential(
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.3),
            nn.Linear(2*128, 3)  # Matches hidden_size=128
        )

    def forward(self, x):
        # Input shape: (batch, 20, 64)
        x = x.permute(0, 2, 1)  # Shape: (batch, 64, 20)
        x = self.convolv(x)      # Shape: (batch, 256, 20)
        x = x.permute(0, 2, 1)   # Shape: (batch, 20, 256)

        lstm_out, (h_n, c_n) = self.lstm(x)  # lstm_out shape: (batch, 20, 128)

        # Compute attention scores
        # Flatten across features: attn_score[i, t] = wT * h_{i, t}
        # Then softmax over t to get α_{i, t}
        attn_scores = self.attn_weight(lstm_out).squeeze(-1)
        attn_weights = torch.softmax(attn_scores, dim=1)
        # Weighted sum of LSTM outputs:
        attn_applied = torch.bmm(attn_weights.unsqueeze(1), lstm_out).squeeze(1)

        # Regression to 3D motion
        output = self.fc(attn_applied)
        return output
    


In [21]:
# Defining Model Training using Training and Validation Sets
def train_model(model, device, train_loader, val_loader=None, epochs=20, model_name="model", example_input=torch.rand(1,20,64), checkpoint_dir="models/"):
    model.to(device)
    criterion = nn.MSELoss()
    optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
    scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=5)
    writer = SummaryWriter(log_dir='runs/' + model_name)
    best_val_loss = float('inf')
    early_stop_counter = 0
    patience = 10 # epochs
    
    # Add the model graph to TensorBoard using example_input
    if example_input is not None:
        writer.add_graph(model, example_input.to(device))

    train_losses = []
    val_losses = []
    r2_scores = []
    
    for epoch in range(epochs):
        # Train
        model.train()
        running_train_loss = 0.0
        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            optimizer.zero_grad()
            preds = model(X_batch)
            loss = criterion(preds, y_batch)
            loss.backward()
            optimizer.step()
            running_train_loss += loss.item() * X_batch.size(0)
        avg_train_loss = running_train_loss / len(train_loader.dataset)
        train_losses.append(avg_train_loss)
        writer.add_scalar("Loss/Train", avg_train_loss, epoch)
        writer.add_scalar("Learning Rate", optimizer.param_groups[0]['lr'], epoch)
            
        if val_loader is not None:
            # Validate
            model.eval()
            running_val_loss = 0.0
            all_preds = []
            all_targets = []
            with torch.no_grad():
                for X_batch, y_batch in val_loader:
                    X_batch = X_batch.to(device)
                    y_batch = y_batch.to(device)
                    preds = model(X_batch)
                    loss = criterion(preds, y_batch)
                    running_val_loss += loss.item() * X_batch.size(0)
                    all_preds.append(preds.cpu())
                    all_targets.append(y_batch.cpu())
            all_preds = torch.cat(all_preds).numpy()
            all_targets = torch.cat(all_targets).numpy()
            r2 = r2_score(all_targets, all_preds)
            r2_scores.append(r2)
            avg_val_loss = running_val_loss / len(val_loader.dataset)
            val_losses.append(avg_val_loss)
            
            # Log to TensorBoard
            
            writer.add_scalar("Loss/Validation", avg_val_loss, epoch)
            writer.add_scalar("R2/Validation", r2, epoch)
            

            print(f"{model_name} Epoch {epoch+1}/{epochs} | Train Loss: {avg_train_loss:.6f} | Val Loss: {avg_val_loss:.6f} | R2: {r2:.6f}")

            scheduler.step(avg_val_loss)

            # Save best model checkpoint
            if avg_val_loss < best_val_loss - 1e-5:
                best_val_loss = avg_val_loss
                early_stop_counter = 0
                print(f"Model Checkpoint | epoch: {epoch} | best_val_loss: {best_val_loss}")
                torch.save(model.state_dict(), checkpoint_dir + model_name + ".pth")
            else:
                early_stop_counter += 1
                if early_stop_counter >= patience:
                    print(f"Early stopping at epoch {epoch+1}")
                    break
    
    writer.close()
    return train_losses, val_losses, r2_scores


In [22]:
"""
Note about the data: each ECOG data file is 15 minutes of 64 channels sampled at 1 KHz of a Rhesus Macaque's brain consisting of the following regions:
 - Motor Cortex
 - Dorsolateral Prefrontal Cortex
 - Ventrolateral Prefrontal Cortex

The wrists were recorded with optical motion capture at 50 KHz.
"""
# Raw data Collection
motion_data_file_l = glob(os.path.join(os.getcwd(), "src/", "motor_cortex/data/data/", "**", "motion*.csv"), recursive=True)
ecog_data_file_l = glob(os.path.join(os.getcwd(), "src/", "motor_cortex/data/data/", "**", "ecog*.csv"), recursive=True)

# SESSION_SET: '/home/linux-pc/gh/CRCNS/src/motor_cortex/data/data/Ipsilateral/2018-05-03_(S3)/X.npy', 4 # This is known from evaluating on several training sets
# best_epoch: best_epoch: 38 | best_val_loss: 0.6026811446545653
# Index of best Session_Set: 22
INDEX = 22
current_ecog_data_file = ecog_data_file_l[INDEX]
current_motion_data_file = motion_data_file_l[INDEX]

print(f"current_ecog_data_file:{current_ecog_data_file}")
print(f"current_motion_data_file:{current_motion_data_file}")

# Process data
# The data is processed
preprocessor = PreprocessData(current_ecog_data_file, current_motion_data_file)
X, y = preprocessor.process()



current_ecog_data_file:/media/linux-pc/Stargate/gh/projects/NeuralNexus/New-Features/Thought-to-Motion/CRCNS/src/motor_cortex/data/data/Ipsilateral/2018-05-03_(S3)/ecog_data.csv
current_motion_data_file:/media/linux-pc/Stargate/gh/projects/NeuralNexus/New-Features/Thought-to-Motion/CRCNS/src/motor_cortex/data/data/Ipsilateral/2018-05-03_(S3)/motion_data.csv


KeyboardInterrupt: 

In [12]:
X.shape

(89999, 20, 64)

In [13]:
y.shape

(89999, 3)

In [ ]:

# Save the processed Data for Later Use
# preprocessor.save()

# # Read in the data
# processed_data_l_X = glob(os.path.join(ROOT_DIR + '/src/motor_cortex/data/data/', '**', "**", "X.npy"))
# processed_data_l_y = glob(os.path.join(ROOT_DIR + '/src/motor_cortex/data/data/', '**', "**", "y.npy"))

# # Define K-fold sets
# test_list_X = []
# train_list_X = []
# test_list_y = []
# train_list_y = []

# for i in range(len(processed_data_l_X)):
#     test_list_X.append(processed_data_l_X[i])
#     test_list_y.append(processed_data_l_y[i])
#     train_X = [x for idx, x in enumerate(processed_data_l_X) if idx != i]
#     train_y = [y for idx, y in enumerate(processed_data_l_y) if idx != i]
#     train_list_X.append(train_X)
#     train_list_y.append(train_y)


# X = np.load(train_list_X[KFOLD][SESSION_SET])
# y = np.load(train_list_y[KFOLD][SESSION_SET])


In [ ]:

# Creating Train and Validation Sets
dataset = EcogMotionDataset(X, y)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_ds, val_ds = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64)

# Assuming train_loader, val_loader, criterion are defined

# 2. CNN_LSTM Hybrid Model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
hybrid_model = EcogToMotionNet()
criterion = nn.MSELoss()
hybrid_train_losses, hybrid_val_losses, hybrid_r2 = train_model(hybrid_model, device, train_loader, val_loader, epochs=100, model_name="Hybrid_CNN_LSTM_ipsilateral_3_output_06_18_2025.pth")


In [ ]:
 
###

# Ipsilateral Model Testing (Real-World Use Case)
# Example usage:
device = torch.device("cuda" if torch.cuda.is_availab0le() else "cpu")
# Raw data Collection
motion_data_file_l = glob(os.path.join(os.getcwd(), "src/", "motor_cortex/data/data/", "**", "motion*.csv"), recursive=True)
ecog_data_file_l = glob(os.path.join(os.getcwd(), "src/", "motor_cortex/data/data/", "**", "ecog*.csv"), recursive=True)



# Define Data to be analyzed
# Contralateral Test data is Session 1
# Contralateral Training Data Best model is Session 3

# Ipsilateral Test Data is Any Session 
# Ipsilateral Training Data Best model is Session 4

# Bilateral Test Data is to be defined...
# Bilateral Training Data is to be defined...

INDEX = 22
current_ecog_data_file = ecog_data_file_l[INDEX]
current_motion_data_file = motion_data_file_l[INDEX]

# Process data
preprocessor = PreprocessData(current_ecog_data_file, current_motion_data_file)
X, y = preprocessor.process()

# Save the processed Data for Later Use
# preprocessor.save()

# Recreate the model structure
hybrid_model = EcogToMotionNet()
hybrid_model.load_state_dict(torch.load("models/Hybrid_CNN_LSTM_ipsilateral_3_output.pth"))
hybrid_model.to(device)
hybrid_model.eval()

# Load the Data
dataset = EcogMotionDataset(X, y)
test_loader = DataLoader(dataset, batch_size=64, shuffle = True)
hybrid_model.to(device)
output_file_path = processed_data_l_X[0].strip("X.npy") + "ecog_predictions.mat"
predictions, targets = predict_and_export(hybrid_model, test_loader, device, output_file_path)
compute_rmse(targets[0], predictions[0])
